##### **Cleaning Data with pandas**

In [34]:
import pandas as pd

##### **1. Load The Dataset**

In [35]:
raw= pd.read_csv('Datasets/kakuma_fresh_mart_sales.csv')

In [36]:
# create a copy of the Data
clean= raw.copy()

clean.head()

,receipt,date,branch,product,category,qty,price,payment,customer
0,R10001,2026-10-01,Kakuma Town,Water,Drinks,2.0,50,M-Pesa,NaN
1,R10002,2026-10-01,LODWAR,Water,Drinks,4.0,50,mpesa,NaN
2,R10003,2026-10-01,Kakuma Town,Salt,Cooking,2.0,40,Cash,NaN
3,R10004,2026-10-01,Kitale,Milk,Dairy,2.0,65,Cash,C0308
4,R10005,01/10/2026,Kakuma Town,Detergent,Household,3.0,270,Cash,NaN


##### **A. Remove Duplicates**

In [37]:
print('Before removing duplicates:', clean.shape)

clean= clean.drop_duplicates()

print('After removing duplicates:', clean.shape)

Before removing duplicates: (2509, 9)
After removing duplicates: (2479, 9)


##### **B. Standardize The Label**

In [38]:
# 1. Branch

# unique branch
clean['branch'].unique()

array(['Kakuma Town', 'LODWAR', 'Kitale', 'Lodwar', 'kitale',
       'kakuma town', 'KITALE', 'KAKUMA TOWN', 'lodwar'], dtype=object)

In [39]:
# remove extra space, convert to title
clean['branch']= clean['branch'].str.strip().str.title()

In [40]:
clean['branch'].unique()

array(['Kakuma Town', 'Lodwar', 'Kitale'], dtype=object)

In [41]:
# category

# unique categories
clean['category'].unique()

array(['Drinks', 'Cooking', 'Dairy', 'Household', 'Grains', 'Produce',
       'produce', 'household', 'drinks', 'DAIRY', 'dairy', 'HOUSEHOLD',
       'COOKING', 'cooking', 'PRODUCE', 'DRINKS', 'GRAINS', 'grains'],
      dtype=object)

In [42]:
# remove extra space, and convert to lower case
clean['category']= clean['category'].str.strip().str.lower()


In [43]:
clean['category'].unique()

array(['drinks', 'cooking', 'dairy', 'household', 'grains', 'produce'],
      dtype=object)

In [44]:
# payment methods
clean['payment'].unique()

array(['M-Pesa', 'mpesa', 'Cash', 'Mpesa', 'Card', 'Credit', nan,
       'M-PESA'], dtype=object)

In [45]:
# rename 'mpesa', 'Mpesa', $ 'M-PESA' to 'M-pesa'
mpesa_map={
    'mpesa': 'M-Pesa', 
     'Mpesa': 'M-Pesa',
     'M-PESA': 'M-Pesa'
}
clean['payment']=clean['payment'].replace(mpesa_map)

In [46]:
clean['payment'].value_counts(dropna= False)

payment
Cash      1105
M-Pesa    1062
Card       170
Credit      81
NaN         61
Name: count, dtype: int64

In [47]:
# Replace missing payment with unknow
clean['payment']=clean['payment'].fillna('unknown')

clean['payment'].value_counts()

payment
Cash       1105
M-Pesa     1062
Card        170
Credit       81
unknown      61
Name: count, dtype: int64

In [48]:
clean.head()

,receipt,date,branch,product,category,qty,price,payment,customer
0,R10001,2026-10-01,Kakuma Town,Water,drinks,2.0,50,M-Pesa,NaN
1,R10002,2026-10-01,Lodwar,Water,drinks,4.0,50,M-Pesa,NaN
2,R10003,2026-10-01,Kakuma Town,Salt,cooking,2.0,40,Cash,NaN
3,R10004,2026-10-01,Kitale,Milk,dairy,2.0,65,Cash,C0308
4,R10005,01/10/2026,Kakuma Town,Detergent,household,3.0,270,Cash,NaN


##### **C. converts Dates and Types**

In [49]:
# unique Dates 
clean['date'].unique()
 

array(['2026-10-01', '01/10/2026', '2026-10-02', '02/10/2026',
       '2026-10-03', '03/10/2026', '2026-10-04', '04/10/2026',
       '2026-10-05', '05/10/2026', '2026-10-06', '06/10/2026',
       '2026-10-07', '07/10/2026', '2026-10-08', '08/10/2026',
       '2026-10-09', '09/10/2026', '2026-10-10', '10/10/2026',
       '2026-10-11', '11/10/2026', '2026-10-12', '12/10/2026',
       '2026-10-13', '13/10/2026', '2026-10-14', '14/10/2026',
       '15/10/2026', '2026-10-15', '2026-10-16', '16/10/2026',
       '2026-10-17', '17/10/2026', '18/10/2026', '2026-10-18',
       '2026-10-19', '19/10/2026', '2026-10-20', '20/10/2026',
       '2026-10-21', '21/10/2026'], dtype=object)

In [50]:
# %Y-%m-%d
iso_dates= pd.to_datetime(clean['date'], format= '%Y-%m-%d', errors= 'coerce' )

iso_dates.sample(10)

2465   2026-10-21
1268   2026-10-11
1742   2026-10-15
100    2026-10-01
2384   2026-10-20
1337   2026-10-11
268    2026-10-03
1377   2026-10-12
1674   2026-10-14
164    2026-10-02
Name: date, dtype: datetime64[ns]

In [51]:
# %d-%m-%Y
day_first= pd.to_datetime(clean['date'], format= '%d-%m-%Y', errors= 'coerce')

day_first.sample(10)

2329   NaT
1070   NaT
246    NaT
311    NaT
254    NaT
494    NaT
1400   NaT
12     NaT
277    NaT
1277   NaT
Name: date, dtype: datetime64[ns]

In [52]:
# fill missing iso_dates with day_first dates
clean['date']=iso_dates.fillna(day_first)
iso_dates.sample()

2067   2026-10-18
Name: date, dtype: datetime64[ns]

##### **D. Handle Missing and impossible value**

In [53]:
# 1. Missing quantites

# drop missing quantites
clean=clean.dropna(subset= 'qty')

clean['qty'].unique()

array([  2.,   4.,   3.,   1.,   6.,   7.,   5.,  -1.,   8.,  -2., 999.,
       500., 250.])

In [54]:
# Price
clean['price'].unique()

array(['50', '40', '65', '270', '320', '70', '190', '85', '45', '250',
       '480', '120', '20', '290', '165', '30', '27', '180', nan, '130',
       '90', '100', '36', '63', '58', '288', '162', '108', 'TBC', '261',
       '148', '76', '432', '171', '81', '225', '18', '243', '117', '0'],
      dtype=object)

In [55]:
# converts price to numeric
clean['price']=pd.to_numeric(clean['price'],errors= 'coerce')

clean['price'].unique()

array([ 50.,  40.,  65., 270., 320.,  70., 190.,  85.,  45., 250., 480.,
       120.,  20., 290., 165.,  30.,  27., 180.,  nan, 130.,  90., 100.,
        36.,  63.,  58., 288., 162., 108., 261., 148.,  76., 432., 171.,
        81., 225.,  18., 243., 117.,   0.])

In [56]:
# Zero prices
clean[clean['price']== 0]

,receipt,date,branch,product,category,qty,price,payment,customer
887,R10874,2026-10-08,Kakuma Town,Soda,drinks,5.0,0.0,M-Pesa,NaN
1174,R11159,2026-10-10,Kakuma Town,Water,drinks,2.0,0.0,Cash,NaN
1874,R11851,2026-10-16,Lodwar,Maize flour,grains,3.0,0.0,Cash,C0139
2048,R12022,2026-10-18,Lodwar,Soda,drinks,5.0,0.0,M-Pesa,NaN
2155,R12128,NaT,Lodwar,Sugar,cooking,2.0,0.0,M-Pesa,NaN


In [57]:
# Replace zero prices with nan
clean.loc[clean['price']== 0, 'price']= float('nan')

In [58]:
clean['price'].unique()

array([ 50.,  40.,  65., 270., 320.,  70., 190.,  85.,  45., 250., 480.,
       120.,  20., 290., 165.,  30.,  27., 180.,  nan, 130.,  90., 100.,
        36.,  63.,  58., 288., 162., 108., 261., 148.,  76., 432., 171.,
        81., 225.,  18., 243., 117.])

In [59]:
# Create 'imputed_price' for missing prices
clean['imputed_price']= clean['price'].isna()

clean.head()

,receipt,date,branch,product,category,qty,price,payment,customer,imputed_price
0,R10001,2026-10-01,Kakuma Town,Water,drinks,2.0,50.0,M-Pesa,NaN,False
1,R10002,2026-10-01,Lodwar,Water,drinks,4.0,50.0,M-Pesa,NaN,False
2,R10003,2026-10-01,Kakuma Town,Salt,cooking,2.0,40.0,Cash,NaN,False
3,R10004,2026-10-01,Kitale,Milk,dairy,2.0,65.0,Cash,C0308,False
4,R10005,NaT,Kakuma Town,Detergent,household,3.0,270.0,Cash,NaN,False


In [60]:
# calculate median price for each product
product_median= clean.groupby('product')['price'].transform('median')

product_median

0        50.0
1        50.0
2        40.0
3        65.0
4       270.0
        ...  
2504     90.0
2505    320.0
2506    320.0
2507    190.0
2508    165.0
Name: price, Length: 2467, dtype: float64

In [61]:
# Replace missing prices with the product median
clean['price']= clean['price'].fillna(product_median)

In [62]:
clean['price'].isna().sum()

np.int64(0)

##### **E. Flag Instead Delete**

In [63]:
# negative units orders
clean['is_return']= clean['qty']< 0

# Unsual over 100 orders
clean['unsual_qty']=clean['qty']> 100

# Missing (empty) customers as new
clean['new_customer']=clean['customer'].isna()

In [64]:
clean.head()

,receipt,date,branch,product,category,qty,price,payment,customer,imputed_price,is_return,unsual_qty,new_customer
0,R10001,2026-10-01,Kakuma Town,Water,drinks,2.0,50.0,M-Pesa,NaN,False,False,False,True
1,R10002,2026-10-01,Lodwar,Water,drinks,4.0,50.0,M-Pesa,NaN,False,False,False,True
2,R10003,2026-10-01,Kakuma Town,Salt,cooking,2.0,40.0,Cash,NaN,False,False,False,True
3,R10004,2026-10-01,Kitale,Milk,dairy,2.0,65.0,Cash,C0308,False,False,False,False
4,R10005,NaT,Kakuma Town,Detergent,household,3.0,270.0,Cash,NaN,False,False,False,True


##### **F. Verify and save**


In [65]:
# Verify
clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2467 entries, 0 to 2508
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   receipt        2467 non-null   object        
 1   date           2269 non-null   datetime64[ns]
 2   branch         2467 non-null   object        
 3   product        2467 non-null   object        
 4   category       2467 non-null   object        
 5   qty            2467 non-null   float64       
 6   price          2467 non-null   float64       
 7   payment        2467 non-null   object        
 8   customer       1099 non-null   object        
 9   imputed_price  2467 non-null   bool          
 10  is_return      2467 non-null   bool          
 11  unsual_qty     2467 non-null   bool          
 12  new_customer   2467 non-null   bool          
dtypes: bool(4), datetime64[ns](1), float64(2), object(6)
memory usage: 202.4+ KB


In [66]:
# save the Dataset
clean.to_csv('Datasets/kakuma_fresh_mart_sales_clean.csv', index= False)

print('Dataset saved successfully!!')

Dataset saved successfully!!
